# Controlador multisemilla para Senior Exp6

Esta notebook corre la workflow base `z635_WorkFlow_01_senior_exp6.ipynb` una vez por cada semilla.
En cada corrida se setean:
- `PARAM$semilla_primigenia` con el valor de la lista `semillas`
- `PARAM$experimento` con un entero incremental en 1
- `PARAM$CN$ratio`
- `PARAM$CN$desvios`
- `PARAM$boruta$aplicacion_boruta`

In [4]:
CONFIG <- list(
  semillas = c(109843, 109847, 109849, 109859, 109873),
  nro_experimento_inicial = 9510L,
  experimento_state_file = "./exp6_multisemilla/ultimo_experimento.txt",
  ratio_canaritos = 0.2,
  desvios_canaritos = 3,
  aplicacion_boruta_flag = FALSE,
  notebook_base = "z635_WorkFlow_01_senior_exp6.ipynb",
  output_dir = "./exp6_multisemilla"
)

In [5]:
leer_notebook <- function(path) {
  nb <- jsonlite::fromJSON(path, simplifyVector = FALSE)
  nb$cells
}

requerir_parametro <- function(nombre, entorno = .GlobalEnv) {
  if (!exists("PARAM", mode = "list", where = entorno, inherits = TRUE)) {
    stop("Falta establecer PARAM en la notebook controladora. Debes definirlo antes de correr la base.", call. = FALSE)
  }
  if (is.null(get("PARAM", envir = entorno, inherits = TRUE)[[nombre]])) {
    stop(paste0("Falta establecer PARAM$", nombre, " en la notebook controladora."), call. = FALSE)
  }
  invisible(get("PARAM", envir = entorno, inherits = TRUE)[[nombre]])
}

requerir_subparametro <- function(grupo, nombre, entorno = .GlobalEnv) {
  if (!exists("PARAM", mode = "list", where = entorno, inherits = TRUE)) {
    stop("Falta establecer PARAM en la notebook controladora. Debe definirse antes de correr la base.", call. = FALSE)
  }
  param_obj <- get("PARAM", envir = entorno, inherits = TRUE)
  if (is.null(param_obj[[grupo]]) || is.null(param_obj[[grupo]][[nombre]])) {
    stop(paste0("Falta establecer PARAM$", grupo, "$", nombre, " en la notebook controladora."), call. = FALSE)
  }
  invisible(param_obj[[grupo]][[nombre]])
}

verificar_parametros <- function(entorno = .GlobalEnv) {
  requerir_parametro("semilla_primigenia", entorno = entorno)
  requerir_parametro("experimento", entorno = entorno)
  requerir_subparametro("CN", "ratio", entorno = entorno)
  requerir_subparametro("CN", "desvios", entorno = entorno)
  requerir_subparametro("boruta", "aplicacion_boruta", entorno = entorno)
  TRUE
}

reemplazar_si_existe <- function(src, patron, reemplazo) {
  if (grepl(patron, src, perl = TRUE)) {
    gsub(patron, reemplazo, src, perl = TRUE)
  } else {
    src
  }
}

ejecutar_corrida <- function(semilla, experimento, ratio, desvios, boruta_flag, base_path, work_dir) {
  dir.create(work_dir, recursive = TRUE, showWarnings = FALSE)
  entorno_corrida <- new.env(parent = globalenv())
  assign("PARAM", list(
    semilla_primigenia = semilla,
    experimento = experimento,
    CN = list(ratio = ratio, desvios = desvios),
    boruta = list(aplicacion_boruta = boruta_flag)
  ), envir = entorno_corrida)

  verificar_parametros(entorno = entorno_corrida)

  inicio_corrida <- Sys.time()
  cat("\n[Inicio corrida] semilla=", semilla, "| experimento=", experimento, "| ratio=", ratio, "| desvios=", desvios, "| boruta=", boruta_flag, "| horario_inicio=", format(inicio_corrida, "%Y-%m-%d %H:%M:%S"), "\n")
  flush.console()

  cells <- leer_notebook(base_path)
  for (i in seq_along(cells)) {
    cell <- cells[[i]]
    if (is.null(cell$cell_type) || !identical(cell$cell_type, "code")) next

    src <- paste(unlist(cell$source), collapse = "")

    src <- reemplazar_si_existe(src, "PARAM\\$semilla_primigenia\\s*<-\\s*.*", paste0("PARAM$semilla_primigenia <- ", semilla))
    src <- reemplazar_si_existe(src, "PARAM\\$experimento\\s*<-\\s*.*", paste0("PARAM$experimento <- ", experimento))
    src <- reemplazar_si_existe(src, "PARAM\\$CN\\$ratio\\s*<-\\s*.*", paste0("PARAM$CN$ratio <- ", ratio))
    src <- reemplazar_si_existe(src, "PARAM\\$CN\\$desvios\\s*<-\\s*.*", paste0("PARAM$CN$desvios <- ", desvios))
    src <- reemplazar_si_existe(src, "PARAM\\$boruta\\$aplicacion_boruta\\s*<-\\s*.*", paste0("PARAM$boruta$aplicacion_boruta <- ", if (boruta_flag) "TRUE" else "FALSE"))

    if (i == 1L) {
      src <- paste0("setwd(", shQuote(work_dir), ")\n", src)
    }

    cat("\n[Celda ", i, " / ", length(cells), "] ejecutando...\n")
    flush.console()
    eval(parse(text = src), envir = entorno_corrida)
    cat("[Celda ", i, " / ", length(cells), "] OK\n")
    flush.console()
  }

  fin_corrida <- Sys.time()
  cat("\n[Fin corrida] semilla=", semilla, "| experimento=", experimento, "| horario_fin=", format(fin_corrida, "%Y-%m-%d %H:%M:%S"), "| duracion_segundos=", round(as.numeric(difftime(fin_corrida, inicio_corrida, units = "secs")), 1), "\n")
  flush.console()

  invisible(NULL)
}



In [6]:
base_path <- normalizePath(file.path(getwd(), CONFIG$notebook_base), winslash = "/")
if (!file.exists(base_path)) stop("No existe la notebook base: ", base_path)

leer_ultimo_experimento <- function(path) {
  if (!file.exists(path)) {
    return(NULL)
  }
  valor <- suppressWarnings(as.integer(readLines(path, warn = FALSE)[1]))
  if (is.na(valor)) {
    stop("El archivo de estado del experimento existe pero no tiene un número válido: ", path, call. = FALSE)
  }
  valor
}

guardar_ultimo_experimento <- function(path, valor) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  writeLines(as.character(valor), con = path, useBytes = TRUE)
}

ultimo_experimento <- leer_ultimo_experimento(CONFIG$experimento_state_file)
if (is.null(ultimo_experimento)) {
  ultimo_experimento <- as.integer(CONFIG$nro_experimento_inicial) - 1L
}

for (i in seq_along(CONFIG$semillas)) {
  ultimo_experimento <- ultimo_experimento + 1L
  semilla <- CONFIG$semillas[i]
  experimento <- ultimo_experimento
  run_dir <- file.path(CONFIG$output_dir, paste0("exp_", experimento, "_s_", semilla))

  cat("\n=== Corrida", i, "de", length(CONFIG$semillas), "===\n")
  cat("semilla:", semilla, "| experimento:", experimento, "| ratio:", CONFIG$ratio_canaritos, "| desvios:", CONFIG$desvios_canaritos, "| boruta:", CONFIG$aplicacion_boruta_flag, "\n")

  PARAM <- NULL

  ejecutar_corrida(
    semilla = semilla,
    experimento = experimento,
    ratio = CONFIG$ratio_canaritos,
    desvios = CONFIG$desvios_canaritos,
    boruta_flag = CONFIG$aplicacion_boruta_flag,
    base_path = base_path,
    work_dir = run_dir
  )

  guardar_ultimo_experimento(CONFIG$experimento_state_file, experimento)
}



=== Corrida 1 de 5 ===
semilla: 109843 | experimento: 9510 | ratio: 0.2 | desvios: 3 | boruta: FALSE 

[Inicio corrida] semilla= 109843 | experimento= 9510 | ratio= 0.2 | desvios= 3 | boruta= FALSE | horario_inicio= 2026-11-02 12:54:46 

[Celda  8  /  125 ] ejecutando...
[Celda  8  /  125 ] OK

[Celda  9  /  125 ] ejecutando...
[Celda  9  /  125 ] OK

[Celda  10  /  125 ] ejecutando...


Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:jsonlite’:

    validate


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘pac

[Celda  10  /  125 ] OK

[Celda  12  /  125 ] ejecutando...
[Celda  12  /  125 ] OK

[Celda  14  /  125 ] ejecutando...
[Celda  14  /  125 ] OK

[Celda  17  /  125 ] ejecutando...
[Celda  17  /  125 ] OK

[Celda  20  /  125 ] ejecutando...


Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




[Celda  20  /  125 ] OK

[Celda  21  /  125 ] ejecutando...
[Celda  21  /  125 ] OK

[Celda  22  /  125 ] ejecutando...
[Celda  22  /  125 ] OK

[Celda  23  /  125 ] ejecutando...
[Celda  23  /  125 ] OK

[Celda  24  /  125 ] ejecutando...
[Celda  24  /  125 ] OK

[Celda  25  /  125 ] ejecutando...
[Celda  25  /  125 ] OK

[Celda  26  /  125 ] ejecutando...
inicio Corregir_Rotas()
fin Corregir_rotas()
[Celda  26  /  125 ] OK

[Celda  28  /  125 ] ejecutando...
[Celda  28  /  125 ] OK

[Celda  29  /  125 ] ejecutando...
[Celda  29  /  125 ] OK

[Celda  30  /  125 ] ejecutando...
[Celda  30  /  125 ] OK

[Celda  31  /  125 ] ejecutando...
[Celda  31  /  125 ] OK

[Celda  32  /  125 ] ejecutando...
[Celda  32  /  125 ] OK

[Celda  33  /  125 ] ejecutando...
[Celda  33  /  125 ] OK

[Celda  34  /  125 ] ejecutando...
[Celda  34  /  125 ] OK

[Celda  35  /  125 ] ejecutando...
[Celda  35  /  125 ] OK

[Celda  36  /  125 ] ejecutando...
[Celda  36  /  125 ] OK

[Celda  37  /  125 ] ejecutand

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




... (4 outputs left out)

[Celda  58  /  125 ] OK

[Celda  59  /  125 ] ejecutando...
[Celda  59  /  125 ] OK

[Celda  60  /  125 ] ejecutando...
[Celda  60  /  125 ] OK

[Celda  61  /  125 ] ejecutando...
inicio AgregaVarRandomForest()
Fin construccion RandomForest
periodo =  201901 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201902 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201903 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201904 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201905 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201906 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  2019

Loading required package: DiceKriging

Loading required package: mlrMBO

Loading required package: mlr

Loading required package: ParamHelpers


Attaching package: ‘ParamHelpers’


The following object is masked from ‘package:R.utils’:

    isVector



Attaching package: ‘mlr’


The following objects are masked from ‘package:R.utils’:

    resample, setThreshold


Loading required package: smoof

Loading required package: checkmate


Attaching package: ‘checkmate’


The following object is masked from ‘package:DiceKriging’:

    checkNames


The following object is masked from ‘package:R.utils’:

    asInt



Attaching package: ‘smoof’


The following objects are masked from ‘package:R.oo’:

    getDescription, getName




[Celda  85  /  125 ] OK

[Celda  87  /  125 ] ejecutando...
[Celda  87  /  125 ] OK

[Celda  88  /  125 ] ejecutando...
[Celda  88  /  125 ] OK

[Celda  90  /  125 ] ejecutando...
[Celda  90  /  125 ] OK

[Celda  92  /  125 ] ejecutando...
[Celda  92  /  125 ] OK

[Celda  94  /  125 ] ejecutando...


Computing y column(s) for design. Not provided.

Mon Nov 02 15:01:52 2026  328, 610, 0.381809495738169, 0.225898351305364, 0.0324920807989492 AUC 0.885775194628583

Mon Nov 02 15:09:11 2026  95, 127, 0.0280044684928923, 0.364930884310306, 0.73406988991639 AUC 0.931820152793095

Mon Nov 02 15:54:29 2026  145, 536, 0.0471123674997164, 0.556420211451572, 1.00225863672382 AUC 0.935244081234258

Mon Nov 02 15:56:53 2026  75, 63, 0.0258060588552728, 0.656514085181973, 0.0745738664779426 AUC 0.926325446139109

Mon Nov 02 16:27:10 2026  37, 1288, 0.235520105934139, 0.262838131848916, 12.3992720474691 AUC 0.926048483590747

Mon Nov 02 16:48:19 2026  391, 88, 0.053473652556886, 0.458948822493864, 0.00126912897490388 AUC 0.931700219769412

Mon Nov 02 16:48:22 2026  5, 7, 0.0862055267752431, 0.723459880626098, 17.225539147077 AUC 0.895980365763691

Mon Nov 02 16:52:47 2026  29, 259, 0.03878791968249, 0.312925407811647, 0.0287229465520684 AUC 0.926303550932456

Mon Nov 02 16:53:35 2026  54, 19, 0.1